# 03 - Modelado de Anomalías Estructurales mediante Autoencoders Variacionales (VAE)

El presente estudio experimental evalúa la eficacia de un **Autoencoder Variacional (VAE)** aplicado a la segmentación y análisis de señales inerciales (acelerometría triaxial espaciotemporal).

Este análisis no pretende consolidar la versión de despliegue, sino abordar rigurosamente las siguientes cuestiones fundamentales:

- Verificar empíricamente la capacidad de la arquitectura para converger hacia una distribución latente que modele fielmente la dinámica estructural en régimen nominal.
- Cuantificar el grado de mejora aportado por la función de puntuación probabilística respecto al modelo referencial estadístico (baseline).
- Optimizar el factor de ponderación del término de divergencia de Kullback-Leibler (KL) frente a la pérdida de reconstrucción determinista.
- Calibrar la frontera de decisión para garantizar una tasa de falsas alarmas que resulte operativa en entornos productivos.
- Examinar la morfología de las señales reconstruidas bajo distintos estados estructurales.

Debe destacarse que el uso de perfiles cinemáticos sintéticos sirve exclusivamente para la validación algorítmica preliminar. La validación conclusiva empleará exclusivamente telemetría extraída directamente del hardware físico de la maqueta de tren.

## Fundamentación Matemática del Modelo

En una arquitectura de Autoencoder tradicional, la propagación genera un mapeo directo y determinista, evaluándose la condición de anomalía a través de una función de pérdida por reconstrucción puramente geométrica:

```text
x -> encoder -> z -> decoder -> x_hat
score = error(x, x_hat)
```

Por el contrario, el enfoque variacional (VAE) impone un prior estocástico sobre la codificación latente, forzando un mapeo continuo y regularizado:

```text
x -> encoder -> mu, logvar -> z ~ N(mu, sigma) -> decoder -> x_hat
```

La función objetivo subyacente (Evidence Lower Bound - ELBO) se aproxima mediante dos componentes antagónicas:

```text
loss = reconstruction_loss + beta * KL(q(z|x) || N(0, I))
```

Bajo el prisma del análisis de vibraciones mecánicas, esta topología implica:

- `reconstruction_loss`: Cuantifica el grado de disonancia vibracional; es decir, hasta qué punto el patrón capturado diverge del régimen dinámico conocido por el decodificador.
- `KL` (Divergencia de Kullback-Leibler): Evalúa la excepcionalidad del estado capturado; penalizando aquellos vectores latentes que se alejan de la distribución normal canónica.
- `beta`: Hiperparámetro que determina la robustez de la regularización.

Para la posterior inferencia y clasificación de anomalías estructurales, se sintetizarán tres funciones de puntuación:

```text
reconstruction_error
kl
vae_score = reconstruction_error + beta * kl
```

In [ ]:
from pathlib import Path
import math
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
def find_data_root():
    candidates = [
        Path("/workspace/inference_server/datos"),
        Path("/workspace/TFM/inference_server/datos"),
        Path.cwd().resolve().parent / "datos",
        Path.cwd().resolve() / "datos",
    ]
    for candidate in candidates:
        if (candidate / "raw" / "train_vibrations_normal.csv").exists():
            return candidate
    raise FileNotFoundError("No se encontro datos/brutos/train_vibrations_normal.csv en rutas esperadas")

DATA_ROOT = find_data_root()
RAW_DIR = DATA_ROOT / "raw"
NORMAL_TRAIN_CSV = RAW_DIR / "train_vibrations_normal.csv"
NORMAL_TEST_CSV = RAW_DIR / "test_vibrations_normal.csv"
ANOMALY_TEST_CSV = RAW_DIR / "test_vibrations_anomaly.csv"

print(f"DATA_ROOT={DATA_ROOT}")
print(f"normal_train={NORMAL_TRAIN_CSV}")
print(f"normal_test={NORMAL_TEST_CSV}")
print(f"anomaly_test={ANOMALY_TEST_CSV}")


In [ ]:
SEED = 42
SAMPLE_RATE_HZ = 100
WINDOW_SIZE = 100
WINDOW_STEP = 50
BATCH_SIZE = 512
EPOCHS = 25
LEARNING_RATE = 1e-3
LATENT_DIM = 16
THRESHOLD_PERCENTILES = [95.0, 97.5, 99.0, 99.5]
MAX_TRAIN_WINDOWS = 12000

# Estudio de ablación simplificado: la anulación paramétrica (beta=0) degrada la arquitectura a un autoencoder estocástico carente de regularización topológica efectiva.
BETA_CANDIDATES = [0.0, 1e-4, 1e-3, 1e-2]
SELECTED_BETA = 1e-3

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE={DEVICE}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
TFM_COLUMNS = ["seq", "timestamp_ms", "acc_x_g", "acc_y_g", "acc_z_g"]
MLOPS_COLUMNS = ["timestamp", "accel_x", "accel_y", "accel_z"]


def read_vibration_csv(path):
    df = pd.read_csv(path)
    cols = set(df.columns)
    if set(TFM_COLUMNS).issubset(cols):
        return df[TFM_COLUMNS].copy()
    if set(MLOPS_COLUMNS).issubset(cols):
        out = pd.DataFrame()
        out["seq"] = np.arange(len(df), dtype=np.int64)
        out["timestamp_ms"] = np.rint(df["timestamp"].astype(float) * 1000.0).astype(np.int64)
        out["acc_x_g"] = df["accel_x"].astype(float)
        out["acc_y_g"] = df["accel_y"].astype(float)
        out["acc_z_g"] = df["accel_z"].astype(float)
        return out
    raise ValueError(f"Formato CSV no soportado: {list(df.columns)}")


def make_windows(values, window_size=100, step=50):
    values = np.asarray(values, dtype=np.float32)
    if values.ndim != 2 or values.shape[1] != 3:
        raise ValueError(f"Expected [samples, 3], got {values.shape}")
    starts = range(0, len(values) - window_size + 1, step)
    return np.asarray([values[i:i + window_size] for i in starts], dtype=np.float32)


def fit_standardizer(windows):
    flat = windows.reshape(-1, windows.shape[-1])
    mean = flat.mean(axis=0)
    std = flat.std(axis=0)
    std = np.where(std < 1e-6, 1.0, std)
    return {"mean": mean.astype(np.float32), "std": std.astype(np.float32)}


def transform_standardizer(windows, scaler):
    return ((windows - scaler["mean"]) / scaler["std"]).astype(np.float32)


def inverse_standardizer(windows_norm, scaler):
    return (windows_norm * scaler["std"] + scaler["mean"]).astype(np.float32)


def to_torch_channels_first(windows):
    return np.transpose(windows, (0, 2, 1)).astype(np.float32)


def to_windows_channels_last(x):
    return np.transpose(x, (0, 2, 1)).astype(np.float32)


In [ ]:
train_df = read_vibration_csv(NORMAL_TRAIN_CSV)
normal_test_df = read_vibration_csv(NORMAL_TEST_CSV)
anomaly_test_df = read_vibration_csv(ANOMALY_TEST_CSV)

train_values = train_df[["acc_x_g", "acc_y_g", "acc_z_g"]].to_numpy(dtype=np.float32)
normal_test_values = normal_test_df[["acc_x_g", "acc_y_g", "acc_z_g"]].to_numpy(dtype=np.float32)
anomaly_test_values = anomaly_test_df[["acc_x_g", "acc_y_g", "acc_z_g"]].to_numpy(dtype=np.float32)

train_windows_raw = make_windows(train_values, WINDOW_SIZE, WINDOW_STEP)
normal_test_windows_raw = make_windows(normal_test_values, WINDOW_SIZE, WINDOW_STEP)
anomaly_test_windows_raw = make_windows(anomaly_test_values, WINDOW_SIZE, WINDOW_STEP)

if len(train_windows_raw) > MAX_TRAIN_WINDOWS:
    idx = np.random.choice(len(train_windows_raw), size=MAX_TRAIN_WINDOWS, replace=False)
    train_windows_raw = train_windows_raw[np.sort(idx)]

scaler = fit_standardizer(train_windows_raw)
train_x = to_torch_channels_first(transform_standardizer(train_windows_raw, scaler))
normal_test_x = to_torch_channels_first(transform_standardizer(normal_test_windows_raw, scaler))
anomaly_test_x = to_torch_channels_first(transform_standardizer(anomaly_test_windows_raw, scaler))

print(f"train_rows={len(train_df):,}")
print(f"normal_test_rows={len(normal_test_df):,}")
print(f"anomaly_test_rows={len(anomaly_test_df):,}")
print(f"train_windows={train_x.shape}")
print(f"normal_test_windows={normal_test_x.shape}")
print(f"anomaly_test_windows={anomaly_test_x.shape}")
print(f"mean={scaler['mean']}")
print(f"std={scaler['std']}")


## Cuantificación frente al Modelo de Referencia Analítico

Cualquier arquitectura neuronal profunda debe justificar su adopción demostrando una superioridad medible sobre aproximaciones estadísticas clásicas. Los parámetros heurísticos evaluados por ventana temporal son los siguientes:

- `vibration_rms`: Raíz cuadrada media de la energía dinámica por eje.
- `mag_std`: Desviación típica del vector espacial de aceleración global.
- `z_std`: Desviación específica vinculada a los desplazamientos verticales (eje Z).
- `composite_rank`: Agregación probabilística fundamentada en los percentiles de los estadísticos anteriores.

Este módulo constituye un marco de control empírico ineludible.

In [ ]:
def window_feature_frame(windows_raw):
    rows = []
    for w in windows_raw:
        centered = w - w.mean(axis=0, keepdims=True)
        magnitude = np.linalg.norm(w, axis=1)
        rows.append({
            "vibration_rms": float(np.sqrt(np.mean(centered**2))),
            "mag_std": float(np.std(magnitude)),
            "x_std": float(np.std(w[:, 0])),
            "y_std": float(np.std(w[:, 1])),
            "z_std": float(np.std(w[:, 2])),
        })
    return pd.DataFrame(rows)


def percentile_score(values, reference):
    reference = np.sort(np.asarray(reference, dtype=np.float64))
    values = np.asarray(values, dtype=np.float64)
    return np.searchsorted(reference, values, side="right") / max(len(reference), 1)

train_base = window_feature_frame(train_windows_raw)
normal_base = window_feature_frame(normal_test_windows_raw)
anomaly_base = window_feature_frame(anomaly_test_windows_raw)

for name in ["vibration_rms", "mag_std", "z_std"]:
    normal_base[f"{name}_rank"] = percentile_score(normal_base[name], train_base[name])
    anomaly_base[f"{name}_rank"] = percentile_score(anomaly_base[name], train_base[name])
    train_base[f"{name}_rank"] = percentile_score(train_base[name], train_base[name])

rank_cols = ["vibration_rms_rank", "mag_std_rank", "z_std_rank"]
train_base["composite_rank"] = train_base[rank_cols].mean(axis=1)
normal_base["composite_rank"] = normal_base[rank_cols].mean(axis=1)
anomaly_base["composite_rank"] = anomaly_base[rank_cols].mean(axis=1)

baseline_rows = []
for score_name in ["vibration_rms", "mag_std", "z_std", "composite_rank"]:
    for percentile in THRESHOLD_PERCENTILES:
        threshold = float(np.percentile(train_base[score_name], percentile))
        normal_ratio = float((normal_base[score_name] > threshold).mean())
        anomaly_ratio = float((anomaly_base[score_name] > threshold).mean())
        baseline_rows.append({
            "model": "baseline",
            "score": score_name,
            "threshold_percentile": percentile,
            "threshold": threshold,
            "normal_anomaly_ratio": normal_ratio,
            "anomaly_anomaly_ratio": anomaly_ratio,
            "ratio_gain": anomaly_ratio / max(normal_ratio, 1e-9),
        })

baseline_results = pd.DataFrame(baseline_rows).sort_values(["threshold_percentile", "ratio_gain"], ascending=[True, False])
display(baseline_results)


In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, in_channels=3, window_size=100, latent_dim=16):
        super().__init__()
        self.in_channels = in_channels
        self.window_size = window_size
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, window_size)
            encoded = self.encoder(dummy)
        self.encoded_shape = tuple(encoded.shape[1:])
        flat_dim = int(np.prod(self.encoded_shape))

        self.fc_mu = nn.Linear(flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(flat_dim, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, flat_dim)

        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(128, 64, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.ConvTranspose1d(64, 32, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.ConvTranspose1d(32, in_channels, kernel_size=7, stride=2, padding=3, output_padding=1),
        )

    def encode(self, x):
        h = self.encoder(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_decode(z).view(-1, *self.encoded_shape)
        out = self.decoder(h)
        if out.size(-1) != self.window_size:
            out = F.interpolate(out, size=self.window_size, mode="linear", align_corners=False)
        return out

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        reconstruction = self.decode(z)
        return reconstruction, mu, logvar


def vae_loss(x, reconstruction, mu, logvar, beta):
    reconstruction_loss = F.mse_loss(reconstruction, x, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return reconstruction_loss + beta * kl, reconstruction_loss, kl


In [ ]:
def split_train_val(x, val_ratio=0.2):
    n = len(x)
    indices = np.random.permutation(n)
    val_n = int(n * val_ratio)
    val_idx = indices[:val_n]
    train_idx = indices[val_n:]
    return x[train_idx], x[val_idx]

train_split, val_split = split_train_val(train_x)
print(f"train_split={train_split.shape}")
print(f"val_split={val_split.shape}")


In [ ]:
def make_loader(x, shuffle=False, batch_size=BATCH_SIZE):
    return DataLoader(TensorDataset(torch.from_numpy(x)), batch_size=batch_size, shuffle=shuffle, drop_last=False)


def train_vae(beta, epochs=EPOCHS, latent_dim=LATENT_DIM):
    model = ConvVAE(in_channels=3, window_size=WINDOW_SIZE, latent_dim=latent_dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    train_loader = make_loader(train_split, shuffle=True)
    val_loader = make_loader(val_split, shuffle=False)
    history = []

    def run_epoch(loader, train=True):
        model.train(train)
        totals = {"loss": 0.0, "recon": 0.0, "kl": 0.0, "n": 0}
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            if train:
                optimizer.zero_grad(set_to_none=True)
            reconstruction, mu, logvar = model(batch)
            loss, recon_loss, kl_loss = vae_loss(batch, reconstruction, mu, logvar, beta=beta)
            if train:
                loss.backward()
                optimizer.step()
            bs = batch.size(0)
            totals["loss"] += float(loss.detach().cpu()) * bs
            totals["recon"] += float(recon_loss.detach().cpu()) * bs
            totals["kl"] += float(kl_loss.detach().cpu()) * bs
            totals["n"] += bs
        return {k: totals[k] / totals["n"] for k in ["loss", "recon", "kl"]}

    start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(train_loader, train=True)
        with torch.no_grad():
            val_metrics = run_epoch(val_loader, train=False)
        row = {
            "epoch": epoch,
            "beta": beta,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        history.append(row)
        print(
            f"beta={beta:g}",
            f"epoch={epoch:03d}",
            f"train_loss={row['train_loss']:.6f}",
            f"val_loss={row['val_loss']:.6f}",
            f"val_recon={row['val_recon']:.6f}",
            f"val_kl={row['val_kl']:.6f}",
        )
    elapsed = time.perf_counter() - start
    return model, pd.DataFrame(history), elapsed

model, history_df, training_seconds = train_vae(SELECTED_BETA)
print(f"training_seconds={training_seconds:.2f}")
print(f"parameters={sum(p.numel() for p in model.parameters()):,}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="Pérdida de Entrenamiento")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="Pérdida de Validación")
axes[0].set_title("Evolución de la Función de Coste")
axes[0].set_xlabel("Épocas")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["val_recon"], label="Reconstrucción (Validación)")
axes[1].plot(history_df["epoch"], history_df["val_kl"], label="Divergencia KL (Validación)")
axes[1].set_title("Descomposición de las Componentes de Pérdida")
axes[1].set_xlabel("Épocas")
axes[1].legend()
plt.show()


In [ ]:
@torch.no_grad()
def score_windows(model, x, beta, batch_size=1024):
    loader = make_loader(x, shuffle=False, batch_size=batch_size)
    rows = []
    model.eval()
    for (batch,) in loader:
        batch = batch.to(DEVICE)
        reconstruction, mu, logvar = model(batch)
        recon_per_window = torch.mean((batch - reconstruction) ** 2, dim=(1, 2))
        kl_per_window = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
        total = recon_per_window + beta * kl_per_window
        rows.append(torch.stack([recon_per_window, kl_per_window, total], dim=1).cpu().numpy())
    scores = np.concatenate(rows, axis=0)
    return pd.DataFrame(scores, columns=["reconstruction_error", "kl", "vae_score"])

train_scores = score_windows(model, train_x, SELECTED_BETA)
normal_scores = score_windows(model, normal_test_x, SELECTED_BETA)
anomaly_scores = score_windows(model, anomaly_test_x, SELECTED_BETA)

summary_rows = []
for score_name in ["reconstruction_error", "kl", "vae_score"]:
    for percentile in THRESHOLD_PERCENTILES:
        threshold = float(np.percentile(train_scores[score_name], percentile))
        normal_ratio = float((normal_scores[score_name] > threshold).mean())
        anomaly_ratio = float((anomaly_scores[score_name] > threshold).mean())
        summary_rows.append({
            "model": "vae",
            "beta": SELECTED_BETA,
            "score": score_name,
            "threshold_percentile": percentile,
            "threshold": threshold,
            "normal_anomaly_ratio": normal_ratio,
            "anomaly_anomaly_ratio": anomaly_ratio,
            "ratio_gain": anomaly_ratio / max(normal_ratio, 1e-9),
        })
vae_results = pd.DataFrame(summary_rows).sort_values(["threshold_percentile", "ratio_gain"], ascending=[True, False])
display(vae_results)


In [ ]:
comparison = pd.concat([
    baseline_results[["model", "score", "threshold_percentile", "threshold", "normal_anomaly_ratio", "anomaly_anomaly_ratio", "ratio_gain"]],
    vae_results[["model", "score", "threshold_percentile", "threshold", "normal_anomaly_ratio", "anomaly_anomaly_ratio", "ratio_gain"]],
], ignore_index=True)

for percentile in THRESHOLD_PERCENTILES:
    print(f"\nPercentil {percentile}")
    display(
        comparison[comparison["threshold_percentile"] == percentile]
        .sort_values("ratio_gain", ascending=False)
        .head(10)
    )


## Estudio de Ablación del Hiperparámetro de Regularización

Esta sección ejecuta un entrenamiento sistemático de múltiples instancias del VAE variando exclusivamente el factor de ponderación `beta`. El objetivo primario es elucidar empíricamente si la imposición del prior probabilístico sobre el espacio latente contribuye de forma medible al poder de discriminación, o si el rendimiento depende unilateralmente del error de reconstrucción.

In [ ]:
RUN_BETA_ABLATION = True
EPOCHS_ABLATION = 8

ablation_rows = []
ablation_histories = []
if RUN_BETA_ABLATION:
    for beta in BETA_CANDIDATES:
        print(f"\n--- beta={beta:g} ---")
        candidate_model, candidate_history, elapsed = train_vae(beta, epochs=EPOCHS_ABLATION, latent_dim=LATENT_DIM)
        candidate_train = score_windows(candidate_model, train_x, beta)
        candidate_normal = score_windows(candidate_model, normal_test_x, beta)
        candidate_anomaly = score_windows(candidate_model, anomaly_test_x, beta)
        candidate_history["elapsed_total"] = elapsed
        ablation_histories.append(candidate_history)
        for score_name in ["reconstruction_error", "kl", "vae_score"]:
            threshold = float(np.percentile(candidate_train[score_name], 99.0))
            normal_ratio = float((candidate_normal[score_name] > threshold).mean())
            anomaly_ratio = float((candidate_anomaly[score_name] > threshold).mean())
            ablation_rows.append({
                "beta": beta,
                "score": score_name,
                "threshold_p99": threshold,
                "normal_anomaly_ratio": normal_ratio,
                "anomaly_anomaly_ratio": anomaly_ratio,
                "ratio_gain": anomaly_ratio / max(normal_ratio, 1e-9),
                "elapsed_seconds": elapsed,
            })

ablation_results = pd.DataFrame(ablation_rows)
if len(ablation_results):
    display(ablation_results.sort_values("ratio_gain", ascending=False))


## Evaluación del Poder Discriminativo y Métricas Binarias

Si bien la telemetría empírica a menudo carecerá de anotaciones detalladas a nivel de milisegundo, la naturaleza controlada del presente conjunto de datos permite plantear un marco de evaluación basado en inferencia binaria clásica (considerando las series en régimen anómalo como eventos positivos y el régimen nominal como negativos).

Aunque esta metodología no es un sustituto estricto para el análisis continuo de las condiciones de contorno del hardware, aporta métricas indispensables (como el F1-Score) para comparar objetivamente la efectividad de los umbrales definidos.

In [ ]:
def classification_metrics(normal_scores_series, anomaly_scores_series, threshold):
    y_true = np.concatenate([
        np.zeros(len(normal_scores_series), dtype=np.int64),
        np.ones(len(anomaly_scores_series), dtype=np.int64),
    ])
    y_pred = np.concatenate([
        np.asarray(normal_scores_series) > threshold,
        np.asarray(anomaly_scores_series) > threshold,
    ]).astype(np.int64)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    specificity = tn / max(tn + fp, 1)
    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1, "specificity": specificity}

metric_rows = []
for score_name in ["reconstruction_error", "kl", "vae_score"]:
    for percentile in THRESHOLD_PERCENTILES:
        threshold = float(np.percentile(train_scores[score_name], percentile))
        metric_rows.append({
            "model": "vae",
            "score": score_name,
            "threshold_percentile": percentile,
            "threshold": threshold,
            **classification_metrics(normal_scores[score_name], anomaly_scores[score_name], threshold),
        })

for score_name in ["vibration_rms", "mag_std", "z_std", "composite_rank"]:
    for percentile in THRESHOLD_PERCENTILES:
        threshold = float(np.percentile(train_base[score_name], percentile))
        metric_rows.append({
            "model": "baseline",
            "score": score_name,
            "threshold_percentile": percentile,
            "threshold": threshold,
            **classification_metrics(normal_base[score_name], anomaly_base[score_name], threshold),
        })

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.sort_values(["f1", "recall"], ascending=False).head(20))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, score_name in zip(axes, ["reconstruction_error", "kl", "vae_score"]):
    threshold = float(np.percentile(train_scores[score_name], 99.0))
    ax.hist(normal_scores[score_name], bins=80, alpha=0.65, label="Test Nominal", density=True)
    ax.hist(anomaly_scores[score_name], bins=80, alpha=0.65, label="Test Anómalo", density=True)
    ax.axvline(threshold, color="red", linestyle="--", label="Frontera (p99 train)")
    ax.set_title(score_name)
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
@torch.no_grad()
def reconstruct_one(model, x, index):
    model.eval()
    batch = torch.from_numpy(x[index:index + 1]).to(DEVICE)
    reconstruction, mu, logvar = model(batch)
    original_norm = batch.cpu().numpy()[0].T
    reconstructed_norm = reconstruction.cpu().numpy()[0].T
    original_raw = inverse_standardizer(original_norm, scaler)
    reconstructed_raw = inverse_standardizer(reconstructed_norm, scaler)
    return original_raw, reconstructed_raw

score_name = "vae_score"
normal_example = int(np.argsort(normal_scores[score_name].to_numpy())[len(normal_scores) // 2])
anomaly_example = int(np.argmax(anomaly_scores[score_name].to_numpy()))

examples = [
    ("Promedio Nominal", normal_test_x, normal_example),
    ("Anomalía Máxima", anomaly_test_x, anomaly_example),
]

fig, axes = plt.subplots(len(examples), 3, figsize=(15, 6), sharex=True)
for row, (title, x, idx) in enumerate(examples):
    orig, rec = reconstruct_one(model, x, idx)
    for col, axis_name in enumerate(["X", "Y", "Z"]):
        ax = axes[row, col]
        ax.plot(orig[:, col], label="Captura", linewidth=1.3)
        ax.plot(rec[:, col], label="Inferencia", linewidth=1.1, alpha=0.85)
        ax.set_title(f"{title} - Eje {axis_name} - Ín: {idx}")
        if row == len(examples) - 1:
            ax.set_xlabel("sample")
        if col == 0:
            ax.set_ylabel("acc")
        ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:
# Análisis longitudinal de la función de puntuación. El avance de ventana obedece a la constante WINDOW_STEP.
anomaly_timeline = anomaly_scores.copy()
anomaly_timeline["window_idx"] = np.arange(len(anomaly_timeline))
anomaly_timeline["start_s"] = anomaly_timeline["window_idx"] * WINDOW_STEP / SAMPLE_RATE_HZ
threshold_p99 = float(np.percentile(train_scores["vae_score"], 99.0))
anomaly_timeline["is_anomaly"] = anomaly_timeline["vae_score"] > threshold_p99

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(anomaly_timeline["start_s"], anomaly_timeline["vae_score"], label="vae_score")
ax.axhline(threshold_p99, color="red", linestyle="--", label="Límite Teórico (p99)")
ax.fill_between(
    anomaly_timeline["start_s"],
    anomaly_timeline["vae_score"],
    threshold_p99,
    where=anomaly_timeline["is_anomaly"],
    color="red",
    alpha=0.25,
    label="Región Anómala Detectada",
)
ax.set_title("Evolución Longitudinal de la Señal Probabilística ante Perturbaciones Físicas")
ax.set_xlabel("Tiempo Continuo (s)")
ax.set_ylabel("vae_score")
ax.legend()
plt.show()


## Directrices de Viabilidad del Modelo Neuronal

La justificación formal para integrar la arquitectura VAE en el pipeline definitivo de inferencia demanda el cumplimiento sistemático de las siguientes métricas de éxito:

1. Confirmación de una separación estadísticamente significativa entre las distribuciones de estados nominales y anómalos.
2. Alta tolerancia y robustez de los ratios frente a ligeras variaciones en la definición de las fronteras de decisión (alta estabilidad del umbral).
3. Superioridad innegable de los estimadores neuronales frente a la combinación probabilística paramétrica base (`composite_rank`).
4. Coherencia cinemática en la decodificación del modelo: elevada fidelidad bajo perfiles conocidos y divergencia estructural inmediata ante frecuencias transitorias anómalas.
5. Excelente granularidad espacial, permitiendo el aislamiento y localización de la fuente del fallo mecánico dentro de un rango discreto de tiempo.

De producirse fallos en las citadas métricas de validación, se plantean los siguientes ejes de optimización arquitectónica:

- Redimensionar el volumen de características comprimidas (`LATENT_DIM`).
- Reevaluar y someter a ajuste fino el parámetro de penalización estocástica (`BETA`).
- Aumentar el factor de solapamiento y/o la longitud de la ventana analizada.
- Aplicar técnicas de Denoising Autoencoder inyectando ruido gaussiano calibrado.
- Transicionar hacia una arquitectura Condicional (CVAE), condicionado a que los datos de telemetría provean factores de entorno fiables (p.ej. posición del hardware sobre el trazado).

## Roadmap Experimental: Migración al Entorno Físico Real

Una vez demostrada la convergencia e idoneidad matemática del framework variacional, el enfoque crítico se desplaza íntegramente hacia la obtención y validación empírica con telemetría del hardware objetivo (ESP32 y acelerómetro de a bordo de la maqueta de tren).

El protocolo estricto de captura para la próxima iteración será el siguiente:

1. Obtención de múltiples secuencias telemétricas en condiciones nominales de la maqueta, en el estado funcional exacto que será exhibido en la demostración final.
2. Almacenamiento riguroso y serializado de los tensores siguiendo el esquema unificado: `seq, timestamp_ms, acc_x_g, acc_y_g, acc_z_g`.
3. Estabilización cinemática: Mantenimiento del régimen de velocidad de translación del vehículo.
4. Trazabilidad absoluta: Elaboración de un registro contextual paralelo contemplando parámetros técnicos de las pruebas (fecha, varianza de alimentación y localización relativa de la sonda inercial).
5. Inyección controlada de fallos estructurales a pequeña escala: juntas degradadas, micro-obstáculos, etc.
6. Aislamiento e integridad del dato: Distinción formal entre cuellos de botella de la red (pérdida de datagramas reflejada en el conteo lógico `seq`) y alteraciones reales de la infraestructura física.

Esta ingesta empírica garantizará el corpus volumétrico necesario para proceder con el re-entrenamiento del peso de la red con un grado de confianza óptimo en su capacidad de delimitación de las patologías espaciales.